# Financial PII Masker

A single class that takes free-text (statements, emails, chat transcripts, support tickets) and can mask financial/personal PII on demand, while still being able to hand back the original text. Nothing is persisted anywhere — it only lives for as long as the object does (session memory only).

**PII categories covered (finance-specific + generic):**
- Person name, phone number, email, address, date of birth / dates
- Credit/debit card number, CVV
- Bank account number, IBAN, IFSC code (India), SWIFT/routing-style codes
- Government IDs: PAN (India), Aadhaar (India), SSN (US), passport number, driver's license
- Customer ID / account reference numbers

Built on Presidio (same library as the other notebooks in this folder) so detection reuses a mature NLP + regex engine instead of hand-rolled parsing.

In [ ]:
# Install Presidio + spaCy English model (skip if already installed in this environment)
!pip install -q presidio_analyzer presidio_anonymizer
!python -m spacy download en_core_web_lg


In [ ]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig


In [ ]:
class FinancialPIIMasker:
    """Wraps one piece of text; mask_data() redacts PII, original_data() returns it unchanged.

    Nothing is written to disk or a database — state lives only on the instance,
    for the lifetime of the Python session.
    """

    # Built-in Presidio entities relevant to finance text
    _BUILTIN_ENTITIES = [
        "PERSON", "PHONE_NUMBER", "EMAIL_ADDRESS", "LOCATION", "DATE_TIME",
        "CREDIT_CARD", "IBAN_CODE", "US_SSN", "US_PASSPORT", "US_DRIVER_LICENSE",
        "US_BANK_NUMBER", "CRYPTO",
    ]

    # (entity_name, regex, score) for things Presidio has no built-in recognizer for.
    # ponytail: regexes are heuristics, not validators (e.g. PAN/Aadhaar don't checksum-verify) —
    # upgrade to a proper checksum recognizer if false positives/negatives start to matter.
    _CUSTOM_PATTERNS = [
        ("PAN_NUMBER", r"\b[A-Z]{5}[0-9]{4}[A-Z]\b", 0.9),
        ("AADHAAR_NUMBER", r"\b\d{4}[ -]?\d{4}[ -]?\d{4}\b", 0.75),
        ("IFSC_CODE", r"\b[A-Z]{4}0[A-Z0-9]{6}\b", 0.9),
        ("CUSTOMER_ID", r"\bCUST-\d{6,10}\b", 0.9),
        ("CVV", r"\bCVV[:\s]*\d{3,4}\b", 0.85),
    ]

    _analyzer = None  # built once, shared across instances (model loading is slow)

    def __init__(self, text: str):
        self._original = text
        self._masked = None
        if FinancialPIIMasker._analyzer is None:
            FinancialPIIMasker._analyzer = self._build_analyzer()
        self._anonymizer = AnonymizerEngine()

    @classmethod
    def _build_analyzer(cls):
        analyzer = AnalyzerEngine()
        for entity, regex, score in cls._CUSTOM_PATTERNS:
            analyzer.registry.add_recognizer(
                PatternRecognizer(
                    supported_entity=entity,
                    patterns=[Pattern(name=f"{entity.lower()}_pattern", regex=regex, score=score)],
                )
            )
        return analyzer

    def mask_data(self) -> str:
        """Return the text with all detected PII redacted."""
        entities = self._BUILTIN_ENTITIES + [p[0] for p in self._CUSTOM_PATTERNS]
        results = self._analyzer.analyze(text=self._original, entities=entities, language="en")
        anonymized = self._anonymizer.anonymize(
            text=self._original,
            analyzer_results=results,
            operators={"DEFAULT": OperatorConfig("replace", {"new_value": "<REDACTED>"})},
        )
        self._masked = anonymized.text
        return self._masked

    def original_data(self) -> str:
        """Return the untouched original text."""
        return self._original


## Demo (fictional data only)

In [ ]:
sample_text = """
Dear Rahul Menon,

Your HDFC credit card ending in 4111 1111 1111 1111 (CVV: 123) was used on 12-Aug-2026.
Customer ID: CUST-78451236. Account IFSC: HDFC0001234. Aadhaar on file: 1234 5678 9012.
PAN: ABCDE1234F. Contact us at +91 98765 43210 or rahul.menon@example.com.
Address on record: 42 MG Road, Ernakulam, Kochi, Kerala 682016.
"""

masker = FinancialPIIMasker(sample_text)

print("MASKED:\n", masker.mask_data())
print("\nORIGINAL (unchanged, still available):\n", masker.original_data())


## Self-check

Minimal assert-based check: masking must actually remove the sensitive substrings, and `original_data()` must stay untouched no matter how many times `mask_data()` is called.

In [ ]:
def demo():
    text = "Contact Rahul Menon at rahul.menon@example.com, card 4111 1111 1111 1111, PAN ABCDE1234F."
    m = FinancialPIIMasker(text)

    masked = m.mask_data()
    assert "rahul.menon@example.com" not in masked
    assert "4111 1111 1111 1111" not in masked
    assert "ABCDE1234F" not in masked

    # original_data must be unaffected by masking, and stable across repeated calls
    assert m.original_data() == text
    m.mask_data()
    assert m.original_data() == text

    print("All self-checks passed.")

demo()
